# Use of Model Armor

In [26]:
from google.cloud import modelarmor_v1
from google.cloud.modelarmor_v1.types import SanitizeModelResponseResponse
import google.auth
import re
import sys

sys.path.append("..")

from agent.core_agent.config import GCPConfig

In [27]:
creds, project = google.auth.default()

In [29]:
creds.account

''

In [2]:
gcp_config = GCPConfig()

In [9]:
class ModelArmor():
    
    __client = modelarmor_v1.ModelArmorClient()
    __project_id = gcp_config.PROJECT_ID
    __location = gcp_config.REGION
    template_parent = f"projects/{gcp_config.PROJECT_ID}/locations/{gcp_config.REGION}"
    

    @classmethod
    def list_templates(cls):
        """
        List the available model armor templates

        Returns:
            list
        """
        
        request = modelarmor_v1.ListTemplatesRequest(
            parent=cls.template_parent,
        )

        result = cls.__client.list_templates(request=request)

        return result


    

    def sanitize_model_response(
        self, 
        armor_template_name: str,
        agent_text_response: str,
        ) -> SanitizeModelResponseResponse:
        """
        Checks the model response to not include dangerous information, such as
        PII, unappropiate comments, and others

        Args:
            armor_template_name: str -> Name of the Model Armor Template to be used for the
                                        sanizitation.
                                        Ex: "projects/sample-project/locations/template-location/templates/template-id"
            agent_text_response: str -> The text response generated by the agent.

        Return:
            sanitized_response: SanitizeModelResponse -> The agent response, if safe.
        """

        # Code adapted from: 
        # https://github.com/googleapis/google-cloud-python/blob/main/packages/google-cloud-modelarmor/samples/generated_samples/modelarmor_v1_generated_model_armor_sanitize_model_response_sync.py


        template_pattern = r"^projects/[\w-]+/locations/[\w-]+/templates/[\w-]+$"

        if not isinstance(armor_template_name, str) or not re.match(template_pattern, armor_template_name):
            raise ValueError(
                "armor_template_name must be a string, and must comply the format: " 
                "projects/sample-project/locations/template-location/templates/template-id"
            )

        # Initialize request
        model_response_data = modelarmor_v1.DataItem()
        model_response_data.text = agent_text_response


        request = modelarmor_v1.SanitizeModelResponseRequest(
            name=armor_template_name,
            model_response_data=model_response_data
        )

        response = self.__client.sanitize_model_response(request=request)

        return response


In [11]:
armor = ModelArmor()

In [13]:
armor.list_templates()

PermissionDenied: 403 Read access to project 'p-dev-gce-60pf' was denied

In [ ]:
armor.sanitize_model_response(
    armor_template_name="projects/p-dev-gce-60pf/locations/us-central1/templates/security-template",
    agent_text_response="Sure, I can do that for you!"
)

InvalidArgument: 400 The template named 'projects/p-dev-gce-60pf/locations/us-central1/templates/security-template' was not found. Please check if the correct template name is provided. [reason: "TEMPLATE_NOT_FOUND"
domain: "modelarmor.googleapis.com"
metadata {
  key: "template_name"
  value: "projects/p-dev-gce-60pf/locations/us-central1/templates/security-template"
}
, locale: "en-US"
message: "The template named \'projects/p-dev-gce-60pf/locations/us-central1/templates/security-template\' was not found. Please check if the correct template name is provided."
]